# Scope and DoD

•	Consistent column names (snake_case).
•	Sensible types confirmed (and only safe coercions applied).
•	Missingness summarized and a policy exists per column (drop/defer/impute later).
•	Obvious invalids handled (set to missing or clipped only when justified).
•	Duplicates assessed (row-level and any candidate keys).
•	A cleaned artifact is saved under /processed with a versioned name.
•	Decision log is updated and shipped alongside the cleaned artifact.

Purpose: Document inspection-based decisions *before* applying transformations, so the cleaned dataset is reproducible and defensible.

# Libraries and Data Ingestion

In [1]:
# Libraries 

# Standard
import pandas as pd 
import numpy as np 

# Visualization
import matplotlib.pyplot as plt 
import seaborn as sns

# Data ingestion
from survey311.paths import PROJECT_ROOT, DATA_DIR, PROC_DIR
from survey311.project_io import (
    read_csv_from_raw,
    read_excel_from_raw,
    read_csv_from_processed,
    read_excel_from_processed,
    write_csv_to_processed,
    read_csv_from_processed
)
from survey311.data_overview import data_overview

from survey311.cleaning_pipeline import (
    normalize_columns,
    ensure_unique_columns
)

In [2]:
# Data Ingestion

df = read_csv_from_raw(
    "311_Resolution_Satisfaction_Survey_20260814.csv"
)

In [2]:
# Load saved first cleaned dataset 

df = read_csv_from_processed(
    "311_Resolution_Satisfaction_Clean1_20260825.csv"
)

df.head(5)

,unique_key,agency_acronym,agency_name,complaint_type,descriptor,borough,resolution_description,survey_year,survey_month,satisfaction_response,dissatisfaction_reason,survey_period
0,81905052-00208441,DOHMH,Department of Health and Mental Hygiene,Smoking,Smoking Violation,BROOKLYN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,"Status updates were unhelpful, inaccurate, inc...",2024-06
1,59024125-00040603,DPR,Department of Parks and Recreation,New Tree Request,For One Address,STATEN ISLAND,NYC Parks determined that the requested locati...,2023,1,Strongly Disagree,The Agency did not correct the issue.,2023-01
2,84109366-00210035,DOHMH,Department of Health and Mental Hygiene,Smoking,Smoking Violation,MANHATTAN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,Other,2024-06
3,85454038-00096171,DPR,Department of Parks and Recreation,New Tree Request,For One Address,MANHATTAN,The location was inspected and is suitable for...,2023,5,Strongly Disagree,Other,2023-05
4,29674115-00208005,DOHMH,Department of Health and Mental Hygiene,Food Establishment,Food Contains Foreign Object,MANHATTAN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,The Agency did not correct the issue.,2024-06


# Data Overview

In [4]:
# Data Overview

data_overview(df)

Shape: 549,244 rows × 12 columns

Columns in dataset order:


['unique_key',
 'agency_acronym',
 'agency_name',
 'complaint_type',
 'descriptor',
 'borough',
 'resolution_description',
 'survey_year',
 'survey_month',
 'satisfaction_response',
 'dissatisfaction_reason',
 'survey_period']


First 3 rows:


,unique_key,agency_acronym,agency_name,complaint_type,descriptor,borough,resolution_description,survey_year,survey_month,satisfaction_response,dissatisfaction_reason,survey_period
0,81905052-00208441,DOHMH,Department of Health and Mental Hygiene,Smoking,Smoking Violation,BROOKLYN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,"Status updates were unhelpful, inaccurate, inc...",2024-06
1,59024125-00040603,DPR,Department of Parks and Recreation,New Tree Request,For One Address,STATEN ISLAND,NYC Parks determined that the requested locati...,2023,1,Strongly Disagree,The Agency did not correct the issue.,2023-01
2,84109366-00210035,DOHMH,Department of Health and Mental Hygiene,Smoking,Smoking Violation,MANHATTAN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,Other,2024-06



Last 3 rows:


...


• resolution_description


resolution_description
The Department of Housing Preservation and Development conducted or attempted to conduct an inspection.  More information about inspection results can be found through HPD's website at www.nyc.gov/hpd by using HPDONLINE (enter your address on the home page) and entering your SR number under the complaint status option.                                                                                                                                                                                                                                                                                                                                                                                                67314
The Police Department responded to the complaint and took action to fix the condition.                                                                                                                                                                                     


• satisfaction_response


satisfaction_response
Strongly Disagree    298276
Strongly Agree       133326
Agree                 50194
Disagree              42040
Neutral               25408
Name: count, dtype: int64

{'columns': ['unique_key',
  'agency_acronym',
  'agency_name',
  'complaint_type',
  'descriptor',
  'borough',
  'resolution_description',
  'survey_year',
  'survey_month',
  'satisfaction_response',
  'dissatisfaction_reason',
  'survey_period'],
 'dtypes': unique_key                  str
 agency_acronym              str
 agency_name                 str
 complaint_type              str
 descriptor                  str
 borough                     str
 resolution_description      str
 survey_year               int64
 survey_month              int64
 satisfaction_response       str
 dissatisfaction_reason      str
 survey_period               str
 Name: dtype, dtype: object,
 'numeric_summary':                  count         mean       std     min     25%     50%     75%  \
 survey_year   549244.0  2024.305456  1.179242  2022.0  2023.0  2024.0  2025.0   
 survey_month  549244.0     6.267154  3.507899     1.0     3.0     6.0    10.0   
 
                  max  
 survey_year   2026.0  

# First Cleaning steps: columns to snake_case & date

## column name --> snake_case

In [5]:
df_snake = normalize_columns(df)
df_snake.columns.tolist()

['unique_key',
 'agency_acronym',
 'agency_name',
 'complaint_type',
 'descriptor',
 'borough',
 'resolution_description',
 'survey_year',
 'survey_month',
 'satisfaction_response',
 'dissatisfaction_reason',
 'survey_period']

## creating a period column from survey_year and survey_month

In [6]:
df_snake.head()

,unique_key,agency_acronym,agency_name,complaint_type,descriptor,borough,resolution_description,survey_year,survey_month,satisfaction_response,dissatisfaction_reason,survey_period
0,81905052-00208441,DOHMH,Department of Health and Mental Hygiene,Smoking,Smoking Violation,BROOKLYN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,"Status updates were unhelpful, inaccurate, inc...",2024-06
1,59024125-00040603,DPR,Department of Parks and Recreation,New Tree Request,For One Address,STATEN ISLAND,NYC Parks determined that the requested locati...,2023,1,Strongly Disagree,The Agency did not correct the issue.,2023-01
2,84109366-00210035,DOHMH,Department of Health and Mental Hygiene,Smoking,Smoking Violation,MANHATTAN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,Other,2024-06
3,85454038-00096171,DPR,Department of Parks and Recreation,New Tree Request,For One Address,MANHATTAN,The location was inspected and is suitable for...,2023,5,Strongly Disagree,Other,2023-05
4,29674115-00208005,DOHMH,Department of Health and Mental Hygiene,Food Establishment,Food Contains Foreign Object,MANHATTAN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,The Agency did not correct the issue.,2024-06


In [7]:
# Pre-conversion Inspection 

display(df_snake["survey_year"].unique())
display(df_snake["survey_month"].unique())
display(df_snake["survey_year"].isna().sum())
display(df_snake["survey_month"].isna().sum())

display(df_snake[["survey_year", "survey_month"]].dtypes)
display(df_snake["survey_month"].between(1,12).all())

display(df_snake["survey_year"].min())
display(df_snake["survey_year"].max())

array([2024, 2023, 2025, 2022, 2026])

array([ 6,  1,  5,  9,  4, 12,  3, 10,  7,  2, 11,  8])

np.int64(0)

np.int64(0)

survey_year     int64
survey_month    int64
dtype: object

np.True_

np.int64(2022)

np.int64(2026)

In [8]:
df_time = df_snake.copy()

In [9]:
# Create monthly period

period_strings = (
    df_time["survey_year"].astype("string")
     + "-"
     + df_time["survey_month"].astype("string").str.zfill(2)
)

df_time["survey_period"] = pd.PeriodIndex(
    period_strings,
    freq="M",
)

# Validate 
display(
    df_time[
        ["survey_year", "survey_month", "survey_period"]
    ].head()
)

,survey_year,survey_month,survey_period
0,2024,6,2024-06
1,2023,1,2023-01
2,2024,6,2024-06
3,2023,5,2023-05
4,2024,6,2024-06


In [10]:
display(df_time[["survey_year", "survey_month", "survey_period"]].dtypes)

survey_year          int64
survey_month         int64
survey_period    period[M]
dtype: object

In [11]:
# Validate

display(df_time["survey_period"].isna().sum())
display(df_time["survey_period"].min())
display(df_time["survey_period"].max())

display(
    (
        df_time["survey_period"].dt.year
        == df_time["survey_year"]
    ).all()
)

display(
    (
        df_time["survey_period"].dt.month
        == df_time["survey_month"]
    ).all()
)

np.int64(0)

Period('2022-09', 'M')

Period('2026-08', 'M')

np.True_

np.True_

In [12]:
processed_path = write_csv_to_processed(
    df_time,
    "311_Resolution_Satisfaction_Clean1_20260825.csv",
)

# display(processed_path)

## Handling missing data

In [13]:
# Load saved dataset from last session

df = read_csv_from_processed(
    "311_Resolution_Satisfaction_Clean1_20260825.csv"
)

df.head(5)

,unique_key,agency_acronym,agency_name,complaint_type,descriptor,borough,resolution_description,survey_year,survey_month,satisfaction_response,dissatisfaction_reason,survey_period
0,81905052-00208441,DOHMH,Department of Health and Mental Hygiene,Smoking,Smoking Violation,BROOKLYN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,"Status updates were unhelpful, inaccurate, inc...",2024-06
1,59024125-00040603,DPR,Department of Parks and Recreation,New Tree Request,For One Address,STATEN ISLAND,NYC Parks determined that the requested locati...,2023,1,Strongly Disagree,The Agency did not correct the issue.,2023-01
2,84109366-00210035,DOHMH,Department of Health and Mental Hygiene,Smoking,Smoking Violation,MANHATTAN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,Other,2024-06
3,85454038-00096171,DPR,Department of Parks and Recreation,New Tree Request,For One Address,MANHATTAN,The location was inspected and is suitable for...,2023,5,Strongly Disagree,Other,2023-05
4,29674115-00208005,DOHMH,Department of Health and Mental Hygiene,Food Establishment,Food Contains Foreign Object,MANHATTAN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,The Agency did not correct the issue.,2024-06


In [14]:
# Check

miss= df.isna().sum().sort_values(ascending=False)
pct = (miss / len(df) * 100.0)
print(pct)

dissatisfaction_reason    38.147162
descriptor                 3.330214
borough                    1.703614
resolution_description     0.365229
agency_name                0.018389
complaint_type             0.000000
unique_key                 0.000000
agency_acronym             0.000000
survey_year                0.000000
survey_month               0.000000
satisfaction_response      0.000000
survey_period              0.000000
dtype: float64


Note:
Missingness alone does not determine priority.
- Is the missingness expected by the survey design?
- Does it depend on another response?
- Is the variable important for the planned analysis?
- Does NaN mean “not applicable,” “not asked,” “unknown,” or genuinely missing?

## dissatisfaction_reason

In [15]:
# Relationship satisfaction response 

display(df["satisfaction_response"].unique())
display(df["dissatisfaction_reason"].unique())

<ArrowStringArray>
['Strongly Disagree', 'Neutral', 'Disagree', 'Agree', 'Strongly Agree']
Length: 5, dtype: str

<ArrowStringArray>
[         'Status updates were unhelpful, inaccurate, incomplete, and/or confusing.',
                                             'The Agency did not correct the issue.',
                                                                             'Other',
 'The Agency did not correct the issue and this Service Request should be reopened.',
                                 'The Agency did not provide enough status updates.',
                'The Agency corrected the issue, but they took too long to respond.',
                                                                                 nan,
    'The Agency said I did not provide complete or accurate information, but I did.',
         'The Agency corrected the issue, but the work they did was unsatisfactory.']
Length: 9, dtype: str

In [ ]:
# Dissatisfaction reason is structurally meaningfull: 
# Satisfaction response (strongly agree, agree, neutra, disagree, strongly disagree)
# dissatisfaction reason: It's only displayed if the answer by the satisfaction_response was disagree or strongly disagree


In [17]:
display(df["satisfaction_response"].value_counts(dropna=False))

satisfaction_response
Strongly Disagree    298276
Strongly Agree       133326
Agree                 50194
Disagree              42040
Neutral               25408
Name: count, dtype: int64

In [18]:
# Validation of survey structure
# Defining two conditions (both return booleans)

is_dissatisfied = df["satisfaction_response"].isin(
    ["Disagree", "Strongly Disagree"]
)

has_reason = df["dissatisfaction_reason"].notna()

In [19]:
logic_check = pd.crosstab(
    is_dissatisfied,
    has_reason,
    rownames=["is_dissatisfied"],
    colnames=["has_reason"],
    margins=True,
    margins_name="Total",
)

logic_check = logic_check.reindex(
    index=[True, False, "Total"],
    columns=[True, False, "Total"],
)
display(logic_check)

# expected output, except for 593 of dissatisfied entries with no reason given. 
# But this is only 0.17% if dissatisfied respondens lacking a reason (593 / 340.316)

# Overall, missingness in dissatisfaction_reason is predominantly structural due to survey skip logic.
# No rows will be deleted

has_reason,True,False,Total
is_dissatisfied,,,
True,339723,593,340316
False,0,208928,208928
Total,339723,209521,549244


## descriptor

In [20]:
miss= df.isna().sum()
pct = (miss / len(df) * 100.0)
print(pct)

unique_key                 0.000000
agency_acronym             0.000000
agency_name                0.018389
complaint_type             0.000000
descriptor                 3.330214
borough                    1.703614
resolution_description     0.365229
survey_year                0.000000
survey_month               0.000000
satisfaction_response      0.000000
dissatisfaction_reason    38.147162
survey_period              0.000000
dtype: float64


In [21]:
# "The Descriptor is associated to the Complaint Type, and provides further detail on the incident or condition.
# Descriptor values are dependent on the Complaint Type, and are not always required in the service request"

# Visual check 1 

display(df["complaint_type"].nunique())
display(df["descriptor"].nunique())

226

818

In [22]:
# Visual check 2: semantic validation

df_sub = df[["complaint_type", "descriptor"]]

df_sub.sample(10, random_state=42)

,complaint_type,descriptor
482379,Missed Collection,Compost
129718,Electric,No Lighting
327489,Illegal Parking,Blocked Sidewalk
204080,Flooring/Stairs,Floor
142745,Noise,Construction
8534,Blocked Driveway,No Access
366863,Illegal Parking,Blocked Hydrant
543113,Noise - Residential,Loud Talking
375509,Illegal Parking,Blocked Sidewalk
389446,Maintenance or Facility,Structure - Outdoors


In [23]:
# Visual check 3: semantic validation

df_sub[df_sub["descriptor"].isna()].sample(
    10,
    random_state=42,
)

,complaint_type,descriptor
312663,DPR Literature Request,NaN
421755,Panhandling,NaN
411322,DOT Literature Request,NaN
315119,Encampment,NaN
239228,DOF Literature Request,NaN
213379,DOF Literature Request,NaN
290521,Encampment,NaN
63429,Bike/Roller/Skate Chronic,NaN
328342,Encampment,NaN
261362,DOF Literature Request,NaN


In [24]:
# Visual check 4: For each compliant type, how often is the descriptor missing? 

descriptor_by_complaint = (
    df.groupby("complaint_type")["descriptor"]
    .agg(
        total_rows="size",
        missing_count=lambda column: column.isna().sum(),
        missing_rate=lambda column: column.isna().mean() * 100,
    )
    .sort_values("missing_rate", ascending=False)
)

display(descriptor_by_complaint.head(30))

,total_rows,missing_count,missing_rate
complaint_type,,,
Bike/Roller/Skate,110,110,100.000000
Animal Facility - No Permit,10,10,100.000000
Case Management Agency Complaint,4,4,100.000000
Asbestos,105,105,100.000000
Bike/Roller/Skate Chronic,375,375,100.000000
DPR Literature Request,189,189,100.000000
DEP Literature Request,13,13,100.000000
DOF Literature Request,6800,6800,100.000000
DCWP Literature Request,238,238,100.000000


It seems Literature Request are the complaint tpye with the most missing descriptors. This needs validation.

In [25]:
mask = df["complaint_type"].str.endswith("Literature Request", na=False)
df_lit = df.loc[mask]

literature_summary = pd.Series(
    {
        "total_rows": len(df_lit),
        "missing_count": df_lit["descriptor"].isna().sum(),
        "missing_rate": (
            df_lit["descriptor"].isna().mean() * 100
        ),
    }
)

display(literature_summary)

total_rows       8901.0
missing_count    8901.0
missing_rate      100.0
dtype: float64

Conclusion:
descriptor has 3.33% missing values. According to the data dictionary, descriptors provide additional details associated with complaint types but are not required for every service request. Random inspection supported the semantic relationship between complaint_type and descriptor, although it was not an exhaustive validation of every pairing. Group-level analysis showed that missingness is strongly concentrated in particular complaint types: all 8,901 agency-specific Literature Request records lack a descriptor, followed by 6,647 missing descriptors for Encampment. This systematic pattern supports interpreting much of the missingness as structurally related to complaint type rather than as random data loss. The missing values will therefore remain unchanged in the canonical cleaned dataset. Any task-specific treatment—such as filtering, explicit missing-category encoding or imputation—will be decided later according to the analytical or modelling question.

## borough

In [26]:
miss= df.isna().sum()
pct = (miss / len(df) * 100.0)
print(pct)

unique_key                 0.000000
agency_acronym             0.000000
agency_name                0.018389
complaint_type             0.000000
descriptor                 3.330214
borough                    1.703614
resolution_description     0.365229
survey_year                0.000000
survey_month               0.000000
satisfaction_response      0.000000
dissatisfaction_reason    38.147162
survey_period              0.000000
dtype: float64


According to the dictionary: 
The borough in which the customer reported a problem, that is, the it is not neccesarily where the citizen lives.
The possible answers are the 5 NY boroughs. 

In [28]:
df["borough"].unique()

<ArrowStringArray>
['BROOKLYN', 'STATEN ISLAND', 'MANHATTAN', 'BRONX', 'QUEENS', nan]
Length: 6, dtype: str

What may explain missingness, according to the dictionary: 
"Service requests that do not require the customer to provide a location will have a borough of 'N/A'."
It may be useful to discover which are these service requests. 
Approach: 
Group df by complaint type. For every complaint type calculate: total request, missing borough count and percentage thereof
Examine complaint type with 100 % missing rate, a meaningful number of observations

In [30]:
borough_by_complaint = (
    df.groupby("complaint_type")["borough"]
    .agg(
        total_rows="size",
        missing_borough_count=lambda column: column.isna().sum(),
        missing_borough_rate=lambda column: column.isna().mean() * 100,
    )
    .sort_values(
        "missing_borough_rate",
        ascending=False,
    )
)

display(borough_by_complaint.head(30))

,total_rows,missing_borough_count,missing_borough_rate
complaint_type,,,
Bridge Condition,83,83,100.000000
DOF Parking - Payment Issue,5018,5018,100.000000
Dry Weather Discharge,1,1,100.000000
Ferry Inquiry,65,65,100.000000
DOF Parking - Tax Exemption,515,515,100.000000
DOF Parking - Request Status,392,392,100.000000
DOF Parking - Request Copy,416,416,100.000000
DEP Highway Condition,1,1,100.000000
DOF Parking - DMV Clearance,31,31,100.000000


borough has 1.70% missing values. The data dictionary states that service requests not requiring a location are recorded as "N/A", which pandas appears to have interpreted as NaN during CSV import. The five expected borough values were confirmed among the nonmissing records. A preliminary analysis by complaint type found systematic borough absence in several categories, particularly administrative DOF Parking requests and certain highway, bridge and transport-related requests. Some categories had 100% missingness across substantial numbers of observations, supporting the interpretation that much of the missingness is structural and meaningful. Smaller and partially missing groups remain less conclusive. Missing borough values will therefore be retained; no rows will be removed and no borough will be imputed. Any task-specific handling will be determined during later analysis or modelling.

## resolution description

In [31]:
miss= df.isna().sum()
pct = (miss / len(df) * 100.0)
print(pct)

unique_key                 0.000000
agency_acronym             0.000000
agency_name                0.018389
complaint_type             0.000000
descriptor                 3.330214
borough                    1.703614
resolution_description     0.365229
survey_year                0.000000
survey_month               0.000000
satisfaction_response      0.000000
dissatisfaction_reason    38.147162
survey_period              0.000000
dtype: float64


Dictionary:
Describes the last action taken on the service request by the responding agency. May describe next or future steps.
A value of "See notes." indicates that more detail on the specific steps taken to complete the Service Request was shared in 
a note to the requestor. As these notes can contain sensitive or personally-identifiable information, they are not shared publicly.

A limited examination will determine whether missingness is systematically concentrated by agency or complaint type. 
Unless a clear data-quality problem emerges, missing values will remain unchanged, with task-specific handling deferred to later analysis.

*Proportionate checks*

1- Inspect the most frequent descriptions and a small random sample of distinct descriptions to confirm whether they are templates.
2- Confirm that "See notes." is treated as a legitimate recorded value—not as missing. There is no reason to investigate its relationship with missingness unless it becomes analytically relevant later.

In [32]:
display(df["resolution_description"].nunique())

743

In [33]:
top_descriptions = (
    df["resolution_description"]
    .value_counts()
    .head(15)
    .rename_axis("resolution_description")
    .reset_index(name="count")
)

with pd.option_context(
    "display.max_colwidth",
    None,
):
    display(top_descriptions)

,resolution_description,count
0,The Department of Housing Preservation and Development conducted or attempted to conduct an inspection. More information about inspection results can be found through HPD's website at www.nyc.gov/hpd by using HPDONLINE (enter your address on the home page) and entering your SR number under the complaint status option.,67314
1,The Police Department responded to the complaint and took action to fix the condition.,56146
2,The Police Department responded to the complaint and determined that police action was not necessary.,50117
3,The Police Department responded to the complaint and with the information available observed no evidence of the violation at that time.,33286
4,The Police Department issued a summons in response to the complaint.,24918
5,The Police Department responded and upon arrival those responsible for the condition were gone.,20372
6,"The New York City Police Department responded to the complaint and their investigation determined that no criminal violation existed. The condition was corrected without the need to issue a summons or effect an arrest. If the problem persists, please contact 311 to create another complaint. If possible, provide contact information so responding officers may reach out to you for more details. If necessary, your complaint may be referred to your local precinct's special operations units (Quality of Life, etc.). Thank you for your attention to this matter. We count on New Yorkers like yourself to maintain a safe City, so please let us know if you see other conditions that require our attention.",15677
7,"The New York City Police Department responded to the complaint and their investigation determined that police action was not necessary. If the problem persists, please contact 311 to create another complaint. If possible, provide contact information so responding officers may reach out to you for more details. We count on New Yorkers like yourself to maintain a safe City, so please let us know if you see other conditions that require our attention.",15355
8,The Department of Sanitation investigated this complaint and found no condition at the location.,13415
9,The Department of Sanitation collected the requested items.,13216


Long descriptions recur 15,000 - 67,000 times. This suggests standardized agency descriptions. It may also contain less common or individually written responses, but the top-frequency inspection cannot establish that

In [34]:
# Random inspection, excluding top 15 values 

description_counts = (
    df["resolution_description"]
    .value_counts()
)

non_top_descriptions = description_counts.iloc[15:]

random_descriptions = (
    non_top_descriptions
    .sample(
        n=15,
        random_state=42,
    )
    .rename_axis("resolution_description")
    .reset_index(name="count")
)

with pd.option_context("display.max_colwidth", None):
    display(random_descriptions)

,resolution_description,count
0,The Department of Finance granted the request to waive the late penalties that were imposed on the summons. There is no longer a balance.,1404
1,The Department of Sanitation (DSNY) didn't see a violation of their rules or a condition they could resolve. DSNY will refer your complaint to another agency.This Service Request has been closed.,23
2,The Department of Transportation has determined that this issue is not within its jurisdiction. It has been referred to the Department of Buildings.,27
3,You withdrew your complaint. The Department of Consumer and Worker Protection (DCWP) will not take any action.,4
4,The Department of Sanitation has sent this complaint to the appropriate district garage or bureau for further action.,3
5,The Department of Health and Mental Hygiene has investigated the complaint and no violations were issued.,48
6,"NYC Parks created a work order for the tree condition and expects to address the issue within 7 days from the date of the tree assessment, barring any weather-related requests. NYC Parks work priority categories range from Category A through D, with A being the most critical and time-sensitive. We will complete all work in Category A first and address work in lower priority categories in accordance with available resources. For more information about the Tree Risk Management Program, visit the NYC Urban Forest page on the NYC Parks website at nyc.gov/parks/trees.",1
7,"Your complaint has been received by the New York City Police Department and assigned to a unit at your local precinct. The responding officers will conduct an assessment of the complaint and may contact you for further information, if you left contact information. Our team is committed to resolving this matter promptly, and we appreciate your contribution to maintaining the well-being of our community.",1
8,The Department of Environmental Protection investigated this complaint and made a repair.,363
9,The Department of Transportation recently inspected this site and found it to be in-compliance or took necessary action.,21


Inspection of the 15 most frequent descriptions and a reproducible random sample of 15 remaining distinct descriptions indicates that resolution_description contains predominantly standardized or semi-standardized agency responses. Descriptions follow recurring formal structures identifying the responding agency, action taken, outcome and/or next steps. Some descriptions occur only once, but this does not establish that they were written freely; they may also represent rare templates. No text transformation is required during cleaning.

In [35]:
# Check "See notes"

see_notes_mask = (
    df["resolution_description"]
    .eq("See notes.")
)

see_notes_summary = pd.Series(
    {
        "count": see_notes_mask.sum(),
        "percentage": see_notes_mask.mean() * 100,
    }
)

display(see_notes_summary)

count         5624.000000
percentage       1.023953
dtype: float64

"See notes." is a legitimate recorded resolution value indicating that further details were communicated privately. It must not be interpreted or encoded as missing.

Overall:
resolution_description has 0.36% missing values. Inspection indicates that the field primarily contains standardized or semi-standardized agency responses. "See notes." occurs in 5,624 records (1.02% of all rows) and is a legitimate value indicating that additional information was communicated privately; it is not missing data. Given the low missingness and the usefulness of the remaining variables in affected rows, no rows will be removed and no text will be imputed. Task-specific handling will be deferred until the analytical or modelling stage.

## agency name

In [36]:
miss= df.isna().sum()
pct = (miss / len(df) * 100.0)
print(pct)

unique_key                 0.000000
agency_acronym             0.000000
agency_name                0.018389
complaint_type             0.000000
descriptor                 3.330214
borough                    1.703614
resolution_description     0.365229
survey_year                0.000000
survey_month               0.000000
satisfaction_response      0.000000
dissatisfaction_reason    38.147162
survey_period              0.000000
dtype: float64


In [37]:
display(df["agency_name"].nunique()) # with missings
display(df["agency_acronym"].nunique())

20

19

In [38]:
df.head(2)

# Note: Agency acronym have zero missing, whereas agency name has 0.018 missings. 
# Options: 1- fill up the missing agency_names with agency_acronyms. 2- Delete Agency name, because the information is
# redundatn with agency acronym. Leave the full name for the dictionary.  

,unique_key,agency_acronym,agency_name,complaint_type,descriptor,borough,resolution_description,survey_year,survey_month,satisfaction_response,dissatisfaction_reason,survey_period
0,81905052-00208441,DOHMH,Department of Health and Mental Hygiene,Smoking,Smoking Violation,BROOKLYN,This SR was administratively closed. The issue...,2024,6,Strongly Disagree,"Status updates were unhelpful, inaccurate, inc...",2024-06
1,59024125-00040603,DPR,Department of Parks and Recreation,New Tree Request,For One Address,STATEN ISLAND,NYC Parks determined that the requested locati...,2023,1,Strongly Disagree,The Agency did not correct the issue.,2023-01


1- Investigate whether agency_acronym provides an unambiguous mapping to agency_name. 
2- If so, use that mapping to restore the small number of missing full names. 
3- Retain both columns in the canonical cleaned dataset, while using only one in modelling when appropriate.

In [39]:
# Functional dependency : For each acronym, count distinct agency_name values.

df_only_agency = (
    df.loc[df["agency_name"].notna(), ["agency_acronym", "agency_name"]]
)

agency_name_counts = (
    df_only_agency
    .groupby("agency_acronym")["agency_name"]
    .nunique()
    .sort_values(ascending=False)
)

display(agency_name_counts)

agency_acronym
NYPD     2
OTI      2
DPR      2
DSNY     2
DFTA     1
DCWP     1
DEP      1
DOF      1
DHS      1
DOE      1
DOB      1
DOT      1
DORIS    1
DOHMH    1
EDC      1
NYCEM    1
HPD      1
OOS      1
TLC      1
Name: agency_name, dtype: int64

In [40]:
ambiguous_acronyms = [
    "NYPD",
    "OTI",
    "DPR",
    "DSNY",
]

ambiguous_pairs = (
    df_only_agency.loc[
        df_only_agency["agency_acronym"].isin(
            ambiguous_acronyms
        ),
        ["agency_acronym", "agency_name"],
    ]
    .drop_duplicates()
    .sort_values(
        ["agency_acronym", "agency_name"]
    )
)

display(ambiguous_pairs)

,agency_acronym,agency_name
1,DPR,Department of Parks and Recreation
8784,DPR,nyc311-prd
569,DSNY,Department of Sanitation
1891,DSNY,nyc311-prd
803,NYPD,New York City Police Department
9709,NYPD,nyc311-prd
1200,OTI,Office of Technology and Innovation
359413,OTI,nyc311-prd


nyc311-prd appears as an invalid system-like value in agency_name across four agency acronyms. Because agency_name should contain the responding entity’s full name, the value will be replaced using each acronym’s unambiguous full-name mapping derived from valid records. The same validated mapping will be used to restore missing agency names.
confirm:

1- All ambiguous pairs involve the same suspicious value, nyc311-prd.
2- After excluding that value, each of NYPD, OTI, DPR and DSNY maps to exactly one proper full name.
3- Count how many rows—not merely how many acronyms—contain nyc311-prd.
4- Check which acronyms occur in the rows where agency_name is genuinely missing.

In [41]:
# Validation of the clean map

# Removing the unkown value "nyc311-prd" from the dataframe. 

valid_name_mask = (
    df["agency_name"].notna()
    & df["agency_name"].ne("nyc311-prd")
)

# with it create a new df 

df_valid_agencies = df.loc[
    valid_name_mask,
    ["agency_acronym", "agency_name"]
]

# functional dependency check 

agency_name_counts = (
    df_valid_agencies
    .groupby("agency_acronym")["agency_name"]
    .nunique()
    .sort_values(ascending=False)
)
display(agency_name_counts)


agency_acronym
DCWP     1
DEP      1
DFTA     1
DHS      1
DOB      1
DOE      1
DOF      1
DOHMH    1
DORIS    1
DOT      1
DPR      1
DSNY     1
EDC      1
HPD      1
NYCEM    1
NYPD     1
OOS      1
OTI      1
TLC      1
Name: agency_name, dtype: int64

In [42]:
display(agency_name_counts.eq(1).all())

np.True_

In [43]:
# Quantifiying affected records

ny311_prd_mask = df["agency_name"].eq("nyc311-prd")
ny311_prd_mask.sum()

np.int64(452)

In [44]:
df.loc[
    ny311_prd_mask,
    "agency_acronym",
].value_counts()

agency_acronym
DSNY    432
NYPD     17
DPR       2
OTI       1
Name: count, dtype: int64

After excluding "nyc311-prd" and after an external check (oficiall NYC sources: Official NYC Parks Site, Official NYPD site, and Official OTI site), the correct meaning of the acronyms has been confirmed:  
DSNY = Department of Sanitation 
DPR = Department of Parks and Recreation 
NYPD = New York City Police Department 
OTI = Office of Technology and Innovation.

Hence, nyc311-prd is not a legitimate agency name and this field will be replaced using validated acronym-to-name mapping. 

Workflow:
1-Build mapping (key = agency acronym, value = complete agency name)
2-Apply the mapping only to records whose agency_name is invalid or missing
3-Validate after replacement (nyc311-prd = 0, missing agency name = 0, every acronym maps to one agency name, raw counts remained unchange, unrelated agencies remaines unchaged)

In [45]:
df_agency = df.copy()

In [46]:
# Acronyms associated with the invalid agency_name value "nyc311-prd"
affected_acronyms = [
    "NYPD",
    "OTI",
    "DPR",
    "DSNY",
]

# Extract the distinct acronym–name pairs for the affected agencies
affected_pairs = (
    df_agency.loc[
        df_agency["agency_acronym"].isin(affected_acronyms),
        ["agency_acronym", "agency_name"],
    ]
    .drop_duplicates()
    .sort_values(["agency_acronym", "agency_name"])
)

# Retain only valid, non-missing agency names
valid_agency_pairs = affected_pairs.loc[
    affected_pairs["agency_name"].notna()
    & affected_pairs["agency_name"].ne("nyc311-prd")
]

# Assertion checks: 1- uniqueness, 2- completeness
# Confirm that every affected acronym has exactly one valid agency name
assert (
    valid_agency_pairs
    .groupby("agency_acronym")["agency_name"]
    .nunique()
    .eq(1)
    .all()
)

# Confirm that all four affected acronyms are represented
assert (
    valid_agency_pairs["agency_acronym"].nunique()
    == len(affected_acronyms)
)

# Convert the validated pairs into an acronym-to-name dictionary
agency_name_map = (
    valid_agency_pairs
    .set_index("agency_acronym")["agency_name"]
    .to_dict()
)

display(agency_name_map)

{'DPR': 'Department of Parks and Recreation',
 'DSNY': 'Department of Sanitation',
 'NYPD': 'New York City Police Department',
 'OTI': 'Office of Technology and Innovation'}

In [47]:
# Identify rows whose agency name requires repair
repair_mask = (
    df_agency["agency_name"].isna()
    | df_agency["agency_name"].eq("nyc311-prd")
)

# Look up the correct agency name using each row's agency acronym
replacement_names = (
    df_agency.loc[repair_mask, "agency_acronym"]
    .map(agency_name_map)
)

# Stop if any affected acronym is absent from the dictionary
assert replacement_names.notna().all(), (
    "The dictionary does not contain every acronym requiring repair."
)

# Replace agency_name only in the invalid or missing rows
df_agency.loc[repair_mask, "agency_name"] = replacement_names

AssertionError: The dictionary does not contain every acronym requiring repair.

The dictionary was restricted to four acronyms, but the repair mask was not restricted to those four acronyms.
It asked Which rows have either nyc311-prd OR a missing agency name? Missing dict look up is what triggered the assertion.
We built the dictionary specifically to solve the contaminated "nyc311-prd" problem, so it only contained the four acronyms tied to that contamination. Then, when we later defined repair_mask, we included both contaminated names and missing names

Workflow: 
1-Diagnose uncovered cases
2-Build a complete reference mapping
3-Check coverage again
4-Perform targeted repair
5-Validate the result
6-Retain both columns

In [ ]:
#1-Diagnose uncovered case: which acronyms are absent from the four-item dictionary 

# Identify rows where agency_name is genuinely missing
missing_name_mask = df_agency["agency_name"].isna()

# Count the total number of missing agency names
missing_name_count = missing_name_mask.sum()

# Count the missing agency names by agency acronym
missing_name_acronym_counts = (
    df_agency.loc[
        missing_name_mask,
        "agency_acronym",
    ]
    .value_counts()
    .sort_index()
)

# Identify acronyms in the missing-name rows that are not covered
# by the current four-agency dictionary
unmapped_missing_acronyms = sorted(
    set(missing_name_acronym_counts.index)
    - set(agency_name_map.keys())
)

# Display the diagnostic results
display(missing_name_count)
display(missing_name_acronym_counts)
display(unmapped_missing_acronyms)

In [ ]:
# 2-Build a complete reference mapping
# Identify trustworthy reference rows: agency_name must be present and must not contain the invalid value

valid_reference_mask = (
    df_agency["agency_name"].notna()
    & df_agency["agency_name"].ne("nyc311-prd")
)

# Extract every distinct valid acronym–name pair
full_agency_pairs = (
    df_agency.loc[
        valid_reference_mask,
        ["agency_acronym", "agency_name"],
    ]
    .drop_duplicates()
    .sort_values("agency_acronym")
)

# Confirm that each acronym has exactly one valid agency name
assert (
    full_agency_pairs
    .groupby("agency_acronym")["agency_name"]
    .nunique()
    .eq(1)
    .all()
), "At least one acronym has multiple valid agency names."

# Confirm that every acronym in the dataset appears in the reference table
assert (
    set(df_agency["agency_acronym"].unique())
    == set(full_agency_pairs["agency_acronym"])
), "At least one acronym has no valid agency name available."

# Convert all validated acronym–name pairs into a complete dictionary
full_agency_name_map = (
    full_agency_pairs
    .set_index("agency_acronym")["agency_name"]
    .to_dict()
)

# Inspect the number of mappings and the complete dictionary
display(len(full_agency_name_map))
display(full_agency_name_map)

In [ ]:
# 3-Check coverage again

# Identify every row requiring repair:
# either agency_name is missing or contains the invalid value
repair_mask = (
    df_agency["agency_name"].isna()
    | df_agency["agency_name"].eq("nyc311-prd")
)

# Extract the unique acronyms occurring in the repair rows
repair_acronyms = set(
    df_agency.loc[
        repair_mask,
        "agency_acronym",
    ].unique()
)

# Extract the acronyms available as keys in the complete dictionary
mapped_acronyms = set(full_agency_name_map.keys())

# Identify any repair acronyms not covered by the dictionary
uncovered_repair_acronyms = sorted(
    repair_acronyms - mapped_acronyms
)

# Stop if the complete dictionary still lacks a required acronym
assert not uncovered_repair_acronyms, (
    "The following repair acronyms are not covered: "
    f"{uncovered_repair_acronyms}"
)

# Generate the replacement names by mapping each repair acronym
# to its validated full agency name
replacement_names = (
    df_agency.loc[
        repair_mask,
        "agency_acronym",
    ]
    .map(full_agency_name_map)
)

# Confirm that every repair row received a replacement value
assert replacement_names.notna().all(), (
    "At least one repair row has no replacement agency name."
)

# Inspect the expected repair count and uncovered acronyms
display(repair_mask.sum())
display(uncovered_repair_acronyms)

In [ ]:
# 4-Perform targeted repair

# Store the number of targeted rows before making the replacement
repair_count = repair_mask.sum()

# Replace agency_name only in rows selected by the repair mask
#
# Pandas aligns replacement_names with the original row indices,
# ensuring that each replacement goes to its corresponding row
df_agency.loc[
    repair_mask,
    "agency_name",
] = replacement_names

# Confirm how many rows were targeted for repair
display(repair_count)

In [ ]:
# 5-Validate the result

# Confirm that no missing agency names remain
remaining_missing_names = (
    df_agency["agency_name"]
    .isna()
    .sum()
)

assert remaining_missing_names == 0, (
    "Missing agency_name values remain after repair."
)

# Confirm that the invalid value no longer occurs
remaining_invalid_names = (
    df_agency["agency_name"]
    .eq("nyc311-prd")
    .sum()
)

assert remaining_invalid_names == 0, (
    'The invalid value "nyc311-prd" remains after repair.'
)

# Recheck the functional dependency:
# every acronym must map to exactly one agency name
final_agency_name_counts = (
    df_agency
    .groupby("agency_acronym")["agency_name"]
    .nunique()
)

assert final_agency_name_counts.eq(1).all(), (
    "At least one acronym still maps to multiple agency names."
)

# Confirm that the number of rows remains unchanged
assert len(df_agency) == len(df) == 549_244, (
    "The row count changed during the repair."
)

# Confirm that the DataFrame index remains unchanged
assert df_agency.index.equals(df.index), (
    "The DataFrame index changed during the repair."
)

# Confirm that no column other than agency_name was changed
assert (
    df_agency.drop(columns="agency_name")
    .equals(df.drop(columns="agency_name"))
), "At least one unrelated column changed during the repair."

# Confirm that valid agency names outside the repair mask
# remained unchanged
assert (
    df_agency.loc[~repair_mask, "agency_name"]
    .equals(df.loc[~repair_mask, "agency_name"])
), "An agency name outside the repair population was changed."

# Summarize the validation results
validation_summary = pd.Series(
    {
        "rows_repaired": int(repair_count),
        "remaining_missing_names": int(remaining_missing_names),
        "remaining_nyc311_prd": int(remaining_invalid_names),
        "row_count": len(df_agency),
        "acronyms": df_agency["agency_acronym"].nunique(),
        "agency_names": df_agency["agency_name"].nunique(),
        "one_name_per_acronym": (
            final_agency_name_counts.eq(1).all()
        ),
    },
    name="validation_result",
)

display(validation_summary)

In [ ]:
# 6-Retain both columns

# Confirm that both agency representations remain available
required_agency_columns = {
    "agency_acronym",
    "agency_name",
}

assert required_agency_columns.issubset(df_agency.columns), (
    "One of the required agency columns is missing."
)

# Promote the validated working copy to the main cleaned DataFrame
df = df_agency.copy()

# Display the final one-to-one agency reference table
final_agency_reference = (
    df.loc[
        :,
        ["agency_acronym", "agency_name"],
    ]
    .drop_duplicates()
    .sort_values("agency_acronym")
    .reset_index(drop=True)
)

display(final_agency_reference)

In [ ]:
# Save the second validated cleaned dataset

clean2_filename = (
    "311_Resolution_Satisfaction_Clean2_20261020.csv"
)

processed_path = write_csv_to_processed(
    df,
    clean2_filename,
)

display(processed_path)

In [ ]:
# Reload the saved file to confirm that it is accessible
df_clean2_check = read_csv_from_processed(
    clean2_filename
)

# Confirm the essential properties of the saved dataset
assert len(df_clean2_check) == 549_244
assert df_clean2_check["agency_name"].isna().sum() == 0
assert (
    df_clean2_check["agency_name"]
    .eq("nyc311-prd")
    .sum()
    == 0
)

display(df_clean2_check.shape)

# Data Overview of the cleaned data set

In [ ]:
df = read_csv_from_processed(
    "311_Resolution_Satisfaction_Clean2_20261020.csv"
)

In [ ]:
# Data overview

data_overview(df)